# Pipeline Generation: Tumor → Brain

Chains both conditional models end-to-end:

1. `novel_segmentation` generates a centered tumor seg map given a size class.
2. The tumor is shifted onto the center-of-mass of a real BraTS segmentation
   (sampled at random), so the spatial placement matches the distribution of
   real tumors instead of always sitting in the middle.
3. `seg2mri` takes the placed seg as cond and generates the brain MRI.

For each pipeline sample we save both the tumor seg (.nii.gz, int16) and the
brain MRI (.nii.gz, float32).

Output: `pipeline/<SAMPLER>/<size>/samples/{tumor,brain}_XXXX.nii.gz` + stats.json


In [ ]:
from pathlib import Path

SAMPLER               = "ddpm"     # "ddim" | "ddpm" | "dpm_solver++" | "unipc"
N_SAMPLES             = 1024
SAMPLER_STEPS         = 250        # full schedule for DDPM; reduce for fast samplers
TUMOR_GUIDANCE_SCALE  = 3.0
BRAIN_GUIDANCE_SCALE  = 3.0
SEED_BASE             = 42
IMAGE_SIZE            = 32
CLASS_ID              = None       # None = cycle 0/1/2 evenly; or 0|1|2 for one fixed class

CHECKPOINT_ROOT = Path(r"C:\Users\Sven\Desktop\diffusion2\FinishedModels\32")
REAL_SEG_DIR    = Path(r"C:\Users\Sven\Desktop\Data\Processed\brats\32_seg")
OUTPUT_ROOT     = Path(rf"pipeline/{SAMPLER.upper()}/{IMAGE_SIZE}")

print(f"sampler={SAMPLER} steps={SAMPLER_STEPS} N={N_SAMPLES} "
      f"guidance=tumor:{TUMOR_GUIDANCE_SCALE} brain:{BRAIN_GUIDANCE_SCALE}")
print(f"class_id={CLASS_ID} (None = cycle through 0/1/2)")


In [ ]:
import sys
import time
import json
from dataclasses import dataclass
from typing import Tuple

import numpy as np
import nibabel as nib
import torch
from scipy.ndimage import shift as nd_shift
from tqdm.auto import tqdm

sys.path.insert(0, str(Path("..") / "Model" / "unconditional"))
sys.path.insert(0, str(Path("..") / "Model" / "conditional"))

from unet3d import UNet3D
from sampler import make_scheduler, sample


# Union Config for both conditional models — torch.load(weights_only=False)
# needs Config in __main__ scope.
@dataclass
class Config:
    data_path:     str = ""
    seg_data_path: str = ""
    labels_csv:    str = ""
    output_path:   str = ""
    image_size:            int = 32
    in_channels:           int = 1
    cond_channels:         int = 0
    model_channels:        int = 96
    channel_mult:          Tuple[int, ...] = (1, 2, 4, 4)
    num_res_blocks:        int = 2
    attention_resolutions: Tuple[int, ...] = (8, 4)
    num_groups:            int = 8
    dropout:               float = 0.0
    num_timesteps:    int = 250
    beta_schedule:    str = "cosine"
    prediction_type:  str = "v"
    num_epochs:     int = 300
    batch_size:     int = 10
    learning_rate:  float = 2e-4
    warmup_steps:   int = 500
    ema_decay:      float = 0.9999
    grad_clip:      float = 0.5
    use_loss_aware_sampling: bool = True
    loss_history_size:       int  = 10
    snr_gamma:               float = 5.0
    cfg_dropout_prob: float = 0.15
    guidance_scale:   float = 3.0
    num_size_classes: int   = 0
    num_seg_classes:  int   = 4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


sys.modules.setdefault("__main__", sys.modules[__name__])
setattr(sys.modules["__main__"], "Config", Config)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")


In [ ]:
def load_model(spec):
    cfg = Config(
        image_size=IMAGE_SIZE,
        attention_resolutions=spec["attention_resolutions"],
        cond_channels=spec["cond_channels"],
        num_size_classes=spec["num_size_classes"],
        prediction_type=spec["prediction_type"],
        beta_schedule=spec["beta_schedule"],
    )
    model = UNet3D(cfg).to(device)
    ckpt_path = CHECKPOINT_ROOT / spec["output_dir"] / "checkpoints" / "final_model.pt"
    ckpt = torch.load(str(ckpt_path), map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    if "ema" in ckpt and "shadow" in ckpt["ema"]:
        for name, p in model.named_parameters():
            if name in ckpt["ema"]["shadow"]:
                p.data.copy_(ckpt["ema"]["shadow"][name].to(device))
    model.eval()
    return model, ckpt_path


TUMOR_SPEC = {
    "output_dir": "novel_segmentation",
    "prediction_type": "v", "beta_schedule": "cosine",
    "attention_resolutions": (8, 4),
    "cond_channels": 0, "num_size_classes": 3,
}
BRAIN_SPEC = {
    "output_dir": "conditional_seg2mri",
    "prediction_type": "v", "beta_schedule": "cosine",
    "attention_resolutions": (8, 4),
    "cond_channels": 1, "num_size_classes": 0,
}

tumor_model, tumor_ckpt = load_model(TUMOR_SPEC)
brain_model, brain_ckpt = load_model(BRAIN_SPEC)

tumor_scheduler = make_scheduler(SAMPLER, TUMOR_SPEC["beta_schedule"], TUMOR_SPEC["prediction_type"])
brain_scheduler = make_scheduler(SAMPLER, BRAIN_SPEC["beta_schedule"], BRAIN_SPEC["prediction_type"])

print(f"loaded tumor model from {tumor_ckpt}")
print(f"loaded brain model from {brain_ckpt}")


In [ ]:
# discretize the tumor model's continuous output back to the 4 BraTS labels
LABEL_VALUES = torch.tensor([-1.0, -1/3, 1/3, 1.0])


def quantize_to_int(volume):
    if volume.dim() > 3:
        volume = volume.squeeze()
    d = torch.abs(volume.unsqueeze(-1) - LABEL_VALUES.to(volume.device))
    return d.argmin(dim=-1).clamp(0, 3).cpu().numpy().astype(np.int16)


def tumor_center_of_mass(lbl):
    mask = lbl > 0
    if not mask.any():
        return np.array([IMAGE_SIZE / 2.0] * 3)
    return np.argwhere(mask).mean(axis=0)


def tumor_bbox(lbl):
    mask = lbl > 0
    if not mask.any():
        return None
    coords = np.argwhere(mask)
    return coords.min(axis=0), coords.max(axis=0)


def clip_shift_to_keep_inside(lbl, shift_vec, vol_size=IMAGE_SIZE):
    # shrink the requested shift so the tumor bbox stays inside [0, vol_size)
    bb = tumor_bbox(lbl)
    if bb is None:
        return np.zeros(3)
    lo, hi = bb
    shift_vec = np.array(shift_vec, dtype=float)
    for axis in range(3):
        shift_vec[axis] = np.clip(shift_vec[axis], -lo[axis], (vol_size - 1) - hi[axis])
    return shift_vec


In [ ]:
print(f"caching real-seg COMs from {REAL_SEG_DIR}...")
real_seg_files = sorted(REAL_SEG_DIR.glob("*.nii.gz"))
assert len(real_seg_files) > 0, f"no real seg files in {REAL_SEG_DIR}"

real_coms = []
for f in tqdm(real_seg_files, desc="real seg COMs"):
    lbl = nib.load(str(f)).get_fdata().astype(np.int16)
    real_coms.append((f.name, tumor_center_of_mass(lbl)))

print(f"cached {len(real_coms)} real-seg COMs")


In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
samples_dir = OUTPUT_ROOT / "samples"
samples_dir.mkdir(parents=True, exist_ok=True)

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

per_sample = []
for i in tqdm(range(N_SAMPLES), desc=f"pipeline ({SAMPLER})"):
    # three independent seeds per sample so tumor/placement/brain noise are decoupled
    tumor_seed = SEED_BASE + i
    placement_seed = SEED_BASE + 10000 + i
    brain_seed = SEED_BASE + 20000 + i

    cls = (i % 3) if CLASS_ID is None else CLASS_ID

    t0 = time.time()

    # 1. generate a centered tumor seg map (CFG-guided)
    size_cond = torch.tensor([cls], device=device, dtype=torch.long)
    tumor_raw = sample(tumor_model, tumor_scheduler, IMAGE_SIZE,
                       num_steps=SAMPLER_STEPS, device=device, seed=tumor_seed,
                       size_cond=size_cond, guidance_scale=TUMOR_GUIDANCE_SCALE)
    tumor_int = quantize_to_int(tumor_raw)
    tumor_time = time.time() - t0

    # 2. shift the tumor to a randomly-chosen real-seg center-of-mass
    rng = np.random.RandomState(placement_seed)
    ref_name, real_com = real_coms[rng.randint(len(real_coms))]
    raw_shift = real_com - tumor_center_of_mass(tumor_int)
    safe_shift = clip_shift_to_keep_inside(tumor_int, raw_shift)
    tumor_placed = nd_shift(tumor_int, safe_shift, order=0, mode="constant", cval=0).astype(np.int16)

    # 3. feed the placed seg into the brain model (renormalize {0..3} -> [-1, 1])
    t1 = time.time()
    cond = torch.from_numpy(tumor_placed).float()
    cond = (cond / 3.0 * 2.0 - 1.0).unsqueeze(0).unsqueeze(0).to(device)
    brain_raw = sample(brain_model, brain_scheduler, IMAGE_SIZE,
                       num_steps=SAMPLER_STEPS, device=device, seed=brain_seed,
                       cond=cond, guidance_scale=BRAIN_GUIDANCE_SCALE)
    brain_time = time.time() - t1
    brain_np = brain_raw.squeeze().cpu().numpy().astype(np.float32)

    # save both
    tumor_path = samples_dir / f"tumor_{i+1:04d}.nii.gz"
    brain_path = samples_dir / f"brain_{i+1:04d}.nii.gz"
    nib.save(nib.Nifti1Image(tumor_placed, np.eye(4)), str(tumor_path))
    nib.save(nib.Nifti1Image(brain_np, np.eye(4)), str(brain_path))

    per_sample.append({
        "tumor_filename":   tumor_path.name,
        "brain_filename":   brain_path.name,
        "size_class":       cls,
        "tumor_seed":       tumor_seed,
        "brain_seed":       brain_seed,
        "placement_ref":    ref_name,
        "placement_shift":  [round(float(v), 3) for v in safe_shift],
        "tumor_voxel_count": int((tumor_placed > 0).sum()),
        "tumor_time_sec":   round(tumor_time, 3),
        "brain_time_sec":   round(brain_time, 3),
        "total_time_sec":   round(tumor_time + brain_time, 3),
        "brain_voxel_mean": round(float(brain_np.mean()), 4),
        "brain_voxel_std":  round(float(brain_np.std()), 4),
    })

peak_gb = round(torch.cuda.max_memory_allocated() / 1e9, 2) if device == "cuda" else None
with (OUTPUT_ROOT / "stats.json").open("w") as f:
    json.dump({
        "sampler":               SAMPLER,
        "steps":                 SAMPLER_STEPS,
        "tumor_guidance_scale":  TUMOR_GUIDANCE_SCALE,
        "brain_guidance_scale":  BRAIN_GUIDANCE_SCALE,
        "class_id":              CLASS_ID,
        "n_samples":             N_SAMPLES,
        "tumor_checkpoint":      str(tumor_ckpt),
        "brain_checkpoint":      str(brain_ckpt),
        "peak_gpu_gb":           peak_gb,
        "per_sample":            per_sample,
    }, f, indent=2)

print(f"\ndone. {N_SAMPLES} pipeline samples -> {samples_dir}")
